# AAPL — 1-minute bars → bronze
---

## Introduction 
Extraction of every field Alpaca exposes for AAPL 1-minute bars, from 2016 to today,
into `lakehouse.bronze` of the Unity Catalog lakehouse.

Fixed decisions (§1 of the design document):

| Decision | Value | Why |
|---|---|---|
| Source | Alpaca Market Data v2, `feed=sip` | 100% of the volume; available on the Basic plan for data older than 15 minutes. |
| Granularity | `1Min` | ~1M rows for AAPL 2016→2026, re-aggregable to 5m/15m/1h/1d without touching the API again. |
| Adjustment | `raw` | Alpaca adjusts with the factors known *today*; storing raw keeps the dataset reproducible. Adjustment happens in silver. |
| Format | Delta Lake, external table in Unity Catalog | Atomic per-partition commits make a retry safe; the catalog makes the same table readable from pandas, Spark and the UC UI. |
| Timezone | UTC | `America/New_York` only appears in silver, where the market calendar makes local time meaningful. |

The layer boundary matters: a bug in a feature definition must never force a
re-download of ten years of history through a 200 req/min budget.

---

## Utils

### Libraries & paths

In [1]:
import logging
import os
import pathlib
import sys

from dotenv import load_dotenv
MAIN = os.path.join(os.path.abspath(os.path.curdir), 'work/')
DATA_ETL_ROOT = os.path.join(MAIN, 'data-etl')
print(DATA_ETL_ROOT)

sys.path.insert(0, str(DATA_ETL_ROOT))

load_dotenv(MAIN)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")

from src.dwh.bronze import alpaca_bars, alpaca_reference
from src.extractions.alpaca import MarketData
from src.storage.catalog import get_catalog

2026-08-10 10:07:20,624 INFO numexpr.utils NumExpr defaulting to 8 threads.


/home/jovyan/work/data-etl


### Parameters

In [ ]:
from datetime import date, timedelta

SYMBOLS = "AAPL"          # the schema is multi-symbol from day one: "AAPL,SPY,QQQ" also works
TIMEFRAME = "1Min"
FEED = "sip"
ADJUSTMENT = "raw"
START_DATE = date(2016, 1, 1)
END_DATE = date.today() - timedelta(days=1) # extract until yesterday, last 15 minutes are not allowed in free plan

CREDENTIALS = {
    "APCA-API-KEY-ID": os.getenv("APCA-API-KEY-ID"),
    "APCA-API-SECRET-KEY": os.getenv("APCA-API-SECRET-KEY"),
}

# UNITY_CATALOG_URI / UNITY_CATALOG_NAME / LAKEHOUSE_ROOT come from the environment.
lakehouse = get_catalog()
market = MarketData(credentials=CREDENTIALS)

print(f"{SYMBOLS} {TIMEFRAME} {START_DATE} \u2192 {END_DATE} (feed={FEED}, adjustment={ADJUSTMENT})")
print(f"catalog {lakehouse.catalog} at {lakehouse.client.uri}, tables under {lakehouse.warehouse_root}")

AAPL 1Min 2026-08-01 → 2026-08-09 (feed=sip, adjustment=raw)
catalog lakehouse at http://unitycatalog:8080, tables under /data/lake


---

## Support tables

### `dim_market_calendar`

Without it there is no way to tell "a minute with no trades" from "the market was
closed", and that distinction decides how gaps are filled in silver. Early closes at
13:00 ET (Thanksgiving eve, 24 Dec) are real and frequent.

In [3]:
calendar = alpaca_reference.ingest_market_calendar(
    market, start=START_DATE, end=END_DATE, lakehouse=lakehouse
)

print(f"{len(calendar)} sessions, {int(calendar['is_half_day'].sum())} of them early closes")
calendar.head()

2026-08-10 10:07:26,194 INFO src.dwh.bronze.alpaca_reference Wrote 2665 trading sessions to file:///data/lake/bronze/dim_market_calendar


2665 sessions, 21 of them early closes


,session_date,open_et,close_et,session_minutes,is_half_day,settlement_date,ingested_at
0,2016-01-04,09:30:00,16:00:00,390,False,2016-01-07,2026-08-10 10:07:26.138109+00:00
1,2016-01-05,09:30:00,16:00:00,390,False,2016-01-08,2026-08-10 10:07:26.138109+00:00
2,2016-01-06,09:30:00,16:00:00,390,False,2016-01-11,2026-08-10 10:07:26.138109+00:00
3,2016-01-07,09:30:00,16:00:00,390,False,2016-01-12,2026-08-10 10:07:26.138109+00:00
4,2016-01-08,09:30:00,16:00:00,390,False,2016-01-13,2026-08-10 10:07:26.138109+00:00


### `dim_corporate_actions`

AAPL in this range: the 4:1 split of 2020-08-31 and quarterly dividends. Both create
discontinuities that would otherwise be learned as signal.

In [4]:
corporate_actions = alpaca_reference.ingest_corporate_actions(
    market, symbols=SYMBOLS, start=START_DATE, end=END_DATE, lakehouse=lakehouse
)

corporate_actions.groupby("type").size()

2026-08-10 10:07:42,901 INFO src.dwh.bronze.alpaca_reference Wrote 43 corporate actions to file:///data/lake/bronze/dim_corporate_actions


type
cash_dividend    42
split             1
dtype: int64

In [5]:
corporate_actions[corporate_actions["type"] == "split"]

,symbol,ex_date,type,ratio,cash_amount,ingested_at
19,AAPL,2020-08-31,split,4.0,NaN,2026-08-10 10:07:42.871454+00:00


## Bronze — `fact_bars_raw`

One request per month (`limit=10000`, `next_page_token`), each write aligned with exactly one
`symbol=/year=/month=` partition. Partitions are rewritten whole rather than appended to,
so a retry after a partial failure cannot duplicate rows.

Set `is_overwrite=False` to resume an interrupted backfill without re-hitting the API.

In [8]:
summary = alpaca_bars.ingest_bars_raw(
    market,
    symbols=SYMBOLS,
    start=START_DATE,
    end=END_DATE,
    timeframe=TIMEFRAME,
    feed=FEED,
    adjustment=ADJUSTMENT,
    lakehouse=lakehouse,
    is_overwrite=True,
)

print(f"{summary['row_count'].sum():,} bars written across {len(summary)} monthly partitions")
summary.tail()

2026-08-10 10:13:26,350 INFO src.dwh.bronze.alpaca_bars Wrote 4572 bars for 2026-08


4,572 bars written across 1 monthly partitions


,year,month,row_count,is_skipped
0,2026,08,4572,False


## Verification

In [9]:
bars = alpaca_bars.read_bars_raw(symbol="AAPL", lakehouse=lakehouse)

print(f"{len(bars):,} rows, {bars['timestamp_at'].min()} \u2192 {bars['timestamp_at'].max()}")
bars.groupby("year").size()

1,906,493 rows, 2016-01-01 00:00:00+00:00 → 2026-08-07 23:59:00+00:00


year
2016    150650
2017    146694
2018    162250
2019    162451
2020    199631
2021    202952
2022    200904
2023    187088
2024    187812
2025    188385
2026    117676
dtype: int64

In [10]:
# Every field the API returns, plus the request provenance that makes an audit possible.
bars.head()

,symbol,timestamp_at,open,high,low,close,volume,trade_count,vwap,feed,adjustment,currency,ingested_at,year,month
0,AAPL,2026-08-03 08:00:00+00:00,309.00,309.105,308.4000,308.66,26055,1626,308.749627,sip,raw,USD,2026-08-10 10:13:26.241968+00:00,2026,08
1,AAPL,2026-08-03 08:01:00+00:00,308.66,309.000,308.2924,308.66,52549,2520,308.706210,sip,raw,USD,2026-08-10 10:13:26.241968+00:00,2026,08
2,AAPL,2026-08-03 08:02:00+00:00,308.80,308.930,308.0100,308.01,13594,567,308.402709,sip,raw,USD,2026-08-10 10:13:26.241968+00:00,2026,08
3,AAPL,2026-08-03 08:03:00+00:00,308.05,308.090,308.0000,308.08,1667,64,308.036684,sip,raw,USD,2026-08-10 10:13:26.241968+00:00,2026,08
4,AAPL,2026-08-03 08:04:00+00:00,308.00,308.190,307.7100,307.71,4784,238,307.785884,sip,raw,USD,2026-08-10 10:13:26.241968+00:00,2026,08


In [11]:
# The logical key (symbol, timestamp_at, feed) must be unique for the ingestion to be idempotent.
duplicate_count = bars.duplicated(subset=["symbol", "timestamp_at", "feed"]).sum()
print(f"duplicates on (symbol, timestamp_at, feed): {duplicate_count}")

duplicates on (symbol, timestamp_at, feed): 0


## Catalog

The three tables are external Delta tables registered in Unity Catalog, so Spark
reads them as `lakehouse.bronze.<table>` without knowing any path.


In [12]:
for table in lakehouse.client.list_tables(lakehouse.catalog, "bronze"):
    print(f"{lakehouse.catalog}.bronze.{table['name']:24} {table['storage_location']}")

lakehouse.bronze.dim_corporate_actions    file:///data/lake/bronze/dim_corporate_actions
lakehouse.bronze.dim_market_calendar      file:///data/lake/bronze/dim_market_calendar
lakehouse.bronze.fact_bars_raw            file:///data/lake/bronze/fact_bars_raw
